In [ ]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio


In [1]:
from fastapi import FastAPI, Query
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import requests
from dotenv import load_dotenv
import os
import spacy
import xml.etree.ElementTree as ET
import json
import requests
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
import os
from label_studio_sdk import LabelStudio
from label_studio_sdk.label_interface.objects import PredictionValue
load_dotenv()
import re
import time
from typing import List, Dict, Any, Tuple, Optional
from PatentProvider import PatentProvider

c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


In [ ]:
import os
import json
import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from label_studio_sdk import LabelStudio

# =====================================================
# CONFIG
# =====================================================

LABEL_STUDIO_URL = "http://localhost:8080"
LABEL_STUDIO_API_KEY = os.getenv("LABEL_STUDIO")
PROJECT_ID = 4

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK")
MODEL_NAME = "deepseek-chat"

BATCH_SIZE = 4
MAX_WORKERS = 15

# MUST match your Label Studio config names
TEXT_FIELD_KEY = "text"           # task.data["text"]
TEXT_TO_NAME = "text"             # <Text name="text" .../>

MENTION_FROM_NAME = "mention"     # <Labels name="mention" toName="text">
REL_FROM_NAME = "relations"       # <Relations name="relations" toName="text">
COREF_REL_VALUE = "COREF"         # <Relation value="COREF"/>

# Your NER mention id pattern: _word + 5 digits (e.g. _apparatus12312)
MENTION_ID_RE = re.compile(r"^_[A-Za-z]+(\d{5})$")

# =====================================================
# CLIENTS
# =====================================================

ls_client = LabelStudio(base_url=LABEL_STUDIO_URL, api_key=LABEL_STUDIO_API_KEY)

project = ls_client.projects.get(id=PROJECT_ID)

# list() returns SDK objects; keep them, but only use .id on them
tasks = list(ls_client.tasks.list(project=project.id))
print("TASKS FOUND:", len(tasks))

create_lock = threading.Lock()

thread_local = threading.local()
def get_openai_client():
    if not hasattr(thread_local, "client"):
        thread_local.client = OpenAI(
            api_key=DEEPSEEK_API_KEY,
            base_url="https://api.deepseek.com",
        )
    return thread_local.client

# =====================================================
# HELPERS (SDK object -> dict)
# =====================================================
def clusters_to_relations(clusters: list[list[str]]) -> list[dict]:
    """
    Convert clusters to relations. We link in a chain to avoid O(n^2).
    [a,b,c] -> a-b, b-c
    """
    rels = []
    for cluster in clusters:
        for i in range(len(cluster) - 1):
            rels.append({
                "from_id": cluster[i],
                "to_id": cluster[i + 1],
                "type": "relation",
                "from_name": REL_FROM_NAME,
                "to_name": TEXT_TO_NAME,
                "value": {"relation": COREF_REL_VALUE},
            })
    return rels

def to_dict(obj):
    """Convert Label Studio SDK model objects to plain dicts safely."""
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    # Pydantic v2
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    # Pydantic v1
    if hasattr(obj, "dict"):
        return obj.dict()
    # Last resort: try vars
    if hasattr(obj, "__dict__"):
        return dict(obj.__dict__)
    raise TypeError(f"Cannot convert {type(obj)} to dict")

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def iter_all_results(task_dict: dict):
    """Yield result items from annotations first, then predictions."""
    for ann in (task_dict.get("annotations") or []):
        ann = to_dict(ann)
        for r in (ann.get("result") or []):
            yield r
    for pred in (task_dict.get("predictions") or []):
        pred = to_dict(pred)
        for r in (pred.get("result") or []):
            yield r

def extract_ner_mentions(task_dict: dict):
    """
    Extract mention spans created by NER with id like _apparatus12312
    Returns a list of result items shaped like LS regions (dicts).
    """
    mentions = []
    seen = set()

    for r in iter_all_results(task_dict):
        # r may already be dict, but be safe
        r = to_dict(r) if not isinstance(r, dict) else r

        if r.get("type") != "labels":
            continue
        if r.get("to_name") != TEXT_TO_NAME:
            continue

        rid = r.get("id")
        if not rid or not MENTION_ID_RE.match(rid):
            continue

        v = r.get("value") or {}
        if "start" not in v or "end" not in v:
            continue
        if not v.get("text"):
            continue
        if not v.get("labels"):
            continue

        if rid in seen:
            continue
        seen.add(rid)

        # Keep only the necessary fields; we’ll re-emit them in our prediction
        mentions.append({
            "id": rid,
            "type": "labels",
            "from_name": MENTION_FROM_NAME,
            "to_name": TEXT_TO_NAME,
            "value": {
                "start": v["start"],
                "end": v["end"],
                "text": v["text"],
                "labels": v["labels"],
            },
        })

    return mentions

# =====================================================
# PROMPT + DEEPSEEK
# =====================================================

def build_coref_prompt(text: str, mentions: list[dict]) -> str:
    mention_rows = []
    for m in mentions:
        v = m["value"]
        mention_rows.append({
            "id": m["id"],
            "text": v.get("text", ""),
            "start": v.get("start"),
            "end": v.get("end"),
            "label": (v.get("labels") or [""])[0],
        })

    return f"""
You are doing COREFERENCE RESOLUTION over pre-marked entity mentions in a patent text.

You are given:
1) The full text.

Task:
- Group mentions that refer to the same real-world entity/concept into clusters (coreference chains).
- Only group truly coreferent mentions (same referent). Do NOT group merely related terms.
- Prefer precision, but do not miss obvious repeats/pronouns/aliases.
- You may omit singleton clusters.

Output rules:
- Return ONLY valid JSON.
- Output must be a JSON array of clusters.
- Each cluster must be a JSON array of the spans 
- Each mention ID can appear in at most one cluster.

TEXT:
\"\"\"{text}\"\"\"

MENTIONS (id, text, start, end, label):
{json.dumps(mention_rows, ensure_ascii=False)}

Return JSON now:
""".strip()

def parse_json_from_model(content: str):
    """
    Robustly extract and parse JSON from LLM output that may include:
    - ```json ... ```
    - ``` ... ```
    - leading/trailing commentary
    """
    if not content:
        raise ValueError("Empty model content")

    s = content.strip()

    # Remove common markdown fences
    # e.g. ```json\n...\n``` or ```\n...\n```
    if s.startswith("```"):
        # Drop first fence line
        first_newline = s.find("\n")
        if first_newline != -1:
            s = s[first_newline + 1:]
        # Drop last fence
        if s.endswith("```"):
            s = s[:-3]
        s = s.strip()

    # If there is still extra text, extract the first top-level JSON array/object.
    # We expect an array for clusters.
    start_candidates = [s.find("["), s.find("{")]
    start_candidates = [i for i in start_candidates if i != -1]
    if not start_candidates:
        raise ValueError(f"No JSON start found in: {content[:200]}")

    start = min(start_candidates)

    # Find matching end by bracket counting (prefer array)
    if s[start] == "[":
        depth = 0
        end = None
        for i in range(start, len(s)):
            ch = s[i]
            if ch == "[":
                depth += 1
            elif ch == "]":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
        if end is None:
            raise ValueError("Unclosed JSON array")
        s = s[start:end]
    else:
        # object case
        depth = 0
        end = None
        for i in range(start, len(s)):
            ch = s[i]
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    end = i + 1
                    break
        if end is None:
            raise ValueError("Unclosed JSON object")
        s = s[start:end]

    return json.loads(s)


def call_deepseek_for_clusters(text: str, mentions: list[dict]) -> list[list[str]]:
    client = get_openai_client()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You cluster mention IDs into coreference chains. Return ONLY JSON."},
            {"role": "user", "content": build_coref_prompt(text, mentions)},
        ],
        temperature=0.0,
    )

    content = response.choices[0].message.content

    try:
        clusters = parse_json_from_model(content)
    except Exception as e:
        print("JSON parse failed:", repr(e))
        print("RAW MODEL OUTPUT (first 400 chars):", (content or "")[:400])
        return []

    if not isinstance(clusters, list):
        return []

    # Normalize, enforce pattern + uniqueness, keep only clusters len>=2
    used = set()
    norm = []
    for c in clusters:
        if not isinstance(c, list):
            continue
        ids = []
        for x in c:
            if isinstance(x, str) and MENTION_ID_RE.match(x) and x not in used:
                ids.append(x)
                used.add(x)
        if len(ids) >= 2:
            norm.append(ids)

    return norm

# =====================================================
# BATCH PROCESSING
# =====================================================

def process_batch(task_batch):
    for task in task_batch:
        task_id = getattr(task, "id", None)
        if task_id is None:
            continue

        # IMPORTANT: tasks.get returns SDK object; convert to dict
        task_obj = ls_client.tasks.get(id=task_id)
        task_dict = to_dict(task_obj)

        text = (task_dict.get("data") or {}).get(TEXT_FIELD_KEY, "")
        if not text:
            print(f"SKIP task {task_id}: no text")
            continue

        mentions = extract_ner_mentions(task_dict)
        if len(mentions) < 2:
            print(f"SKIP task {task_id}: <2 NER mentions with _word##### ids")
            continue

        print(f"CALLING COREF API for task {task_id} (mentions={len(mentions)})")

        try:
            clusters = call_deepseek_for_clusters(text, mentions)
            relations = clusters_to_relations(clusters)

            # Label Studio needs the mention regions present in the SAME result payload
            result = mentions + relations

            with create_lock:
                ls_client.predictions.create(
                    task=task_id,
                    model_version="deepseek-coref",
                    score=1.0,
                    result=result,
                )

            print(f"CREATED COREF prediction for task {task_id}: clusters={len(clusters)} relations={len(relations)}")

        except Exception as e:
            print("ERROR task", task_id, "->", repr(e))

# =====================================================
# RUN
# =====================================================

batches = list(chunked(tasks, BATCH_SIZE))
print("BATCHES:", len(batches))

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(process_batch, b) for b in batches]
    print("FUTURES SUBMITTED:", len(futures))

    for i, fut in enumerate(as_completed(futures), 1):
        fut.result()
        if i % 25 == 0:
            print("DONE", i, "/", len(futures))


TASKS FOUND: 11
BATCHES: 3
FUTURES SUBMITTED: 3
CALLING COREF API for task 27024 (mentions=11)


c:\Users\Caleb\Documents\LLM Patent Claim Generator Thesis\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='annotations', input_value=[], input_type=list])
  return self.__pydantic_serializer__.to_python(


CALLING COREF API for task 27028 (mentions=56)
CALLING COREF API for task 27032 (mentions=27)
CREATED COREF prediction for task 27024: clusters=5 relations=5
CALLING COREF API for task 27025 (mentions=26)
CREATED COREF prediction for task 27032: clusters=3 relations=4
CALLING COREF API for task 27033 (mentions=26)
CREATED COREF prediction for task 27025: clusters=2 relations=2
CALLING COREF API for task 27026 (mentions=26)
CREATED COREF prediction for task 27028: clusters=27 relations=29
CALLING COREF API for task 27029 (mentions=21)
CREATED COREF prediction for task 27033: clusters=0 relations=0
CREATED COREF prediction for task 27026: clusters=10 relations=15
CALLING COREF API for task 27027 (mentions=11)
CREATED COREF prediction for task 27029: clusters=0 relations=0
CALLING COREF API for task 27034 (mentions=307)
CALLING COREF API for task 27030 (mentions=55)
ERROR task 27034 -> BadRequestError('Error code: 400 - {\'error\': {\'message\': "This model\'s maximum context length is 13